In [1]:
from unstructured.partition.pdf import partition_pdf

c:\Users\tchouarr\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
elements = partition_pdf(
    filename="..\data\[GU] Calcul des besoins V12.pdf",
    strategy="fast",
    extract_images_in_pdf=False,
    languages=["fra"]
)

In [7]:
len(elements)

247

In [8]:
elements[:5]

## cheking metadata

In [9]:
for el in elements[:50]:
    print(type(el))
    print(el.text)
    print(el.metadata)
    print("=" * 50)

<class 'unstructured.documents.elements.Title'>
SAGE X3 – V12 – CALCUL DES BESOINS
<class 'unstructured.documents.elements.Title'>
Guide Utilisateur
<class 'unstructured.documents.elements.Text'>
Version du 22/05/2024
<class 'unstructured.documents.elements.Footer'>
1
<class 'unstructured.documents.elements.Title'>
HISTORIQUE DU DOCUMENT
<class 'unstructured.documents.elements.Title'>
Date
<class 'unstructured.documents.elements.Title'>
BZ
<class 'unstructured.documents.elements.Title'>
Chapitre
<class 'unstructured.documents.elements.Title'>
Descriptif
<class 'unstructured.documents.elements.Text'>
20/06/2011
<class 'unstructured.documents.elements.Title'>
Version initiale
<class 'unstructured.documents.elements.Text'>
19/08/2011
<class 'unstructured.documents.elements.Text'>
27/09/2018
<class 'unstructured.documents.elements.Text'>
2.1, 2.4, 2.5, 3.3
<class 'unstructured.documents.elements.ListItem'>
Possibilité de calculer le besoin selon un seuil de réappro manuel avec besoin = seu

In [10]:
markdown_text = ""

for el in elements:

    page = getattr(el.metadata, "page_number", None)

    if el.category == "Title":
        markdown_text += f"\n# {el.text}\n"

    else:
        markdown_text += f"\n{el.text}\n"

In [11]:
markdown_text

'\n# SAGE X3 – V12 – CALCUL DES BESOINS\n\n# Guide Utilisateur\n\nVersion du 22/05/2024\n\n1\n\n# HISTORIQUE DU DOCUMENT\n\n# Date\n\n# BZ\n\n# Chapitre\n\n# Descriptif\n\n20/06/2011\n\n# Version initiale\n\n19/08/2011\n\n27/09/2018\n\n2.1, 2.4, 2.5, 3.3\n\nPossibilité de calculer le besoin selon un seuil de réappro manuel avec besoin = seuil réappro - Possibilité de calculer le besoin d’un site en prenant en compte les données de plusieurs sites (groupe de site). Mise à jour Guide en Version 11\n\nPossibilité de calculer le besoin selon un seuil de réappro manuel avec besoin = seuil réappro - Possibilité de calculer le besoin d’un site en prenant en compte les données de plusieurs sites (groupe de site). Mise à jour Guide en Version 11\n\n22/05/2024\n\n# MAJ EN V12\n\nMAJ charte graphique et format\n\n2\n\n# Rédacteur\n\n# P. LEROY\n\n# J. COURIC\n\n# D. CARDANTE\n\n# B. LIFY\n\n# G. BOURRILLON\n\n# SOMMAIRE\n\n1 PARAMETRAGE ..................................................... 4 1.1 

In [15]:
elements

In [12]:
from langchain_core.documents import Document

docs = []

current_header = "Sans titre"

for el in elements:

    if el.category == "Title":
        current_header = el.text
        continue

    text = el.text.strip()

    if not text:
        continue

    page = getattr(el.metadata, "page_number", None)

    doc = Document(
        page_content=text,
        metadata={
            "header": current_header,
            "page": page,
            "category": el.category,
            "source": el.metadata.filename
        }
    )

    docs.append(doc)

In [13]:
docs

[Document(metadata={'header': 'Guide Utilisateur', 'page': 1, 'category': 'UncategorizedText', 'source': '[GU] Calcul des besoins V12.pdf'}, page_content='Version du 22/05/2024'),
 Document(metadata={'header': 'Guide Utilisateur', 'page': 1, 'category': 'Footer', 'source': '[GU] Calcul des besoins V12.pdf'}, page_content='1'),
 Document(metadata={'header': 'Descriptif', 'page': 2, 'category': 'UncategorizedText', 'source': '[GU] Calcul des besoins V12.pdf'}, page_content='20/06/2011'),
 Document(metadata={'header': 'Version initiale', 'page': 2, 'category': 'UncategorizedText', 'source': '[GU] Calcul des besoins V12.pdf'}, page_content='19/08/2011'),
 Document(metadata={'header': 'Version initiale', 'page': 2, 'category': 'UncategorizedText', 'source': '[GU] Calcul des besoins V12.pdf'}, page_content='27/09/2018'),
 Document(metadata={'header': 'Version initiale', 'page': 2, 'category': 'UncategorizedText', 'source': '[GU] Calcul des besoins V12.pdf'}, page_content='2.1, 2.4, 2.5, 3.3'

In [14]:
len(docs)

163

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

In [17]:
final_docs = splitter.split_documents(docs)

In [18]:
final_docs

[Document(metadata={'header': 'Guide Utilisateur', 'page': 1, 'category': 'UncategorizedText', 'source': '[GU] Calcul des besoins V12.pdf'}, page_content='Version du 22/05/2024'),
 Document(metadata={'header': 'Guide Utilisateur', 'page': 1, 'category': 'Footer', 'source': '[GU] Calcul des besoins V12.pdf'}, page_content='1'),
 Document(metadata={'header': 'Descriptif', 'page': 2, 'category': 'UncategorizedText', 'source': '[GU] Calcul des besoins V12.pdf'}, page_content='20/06/2011'),
 Document(metadata={'header': 'Version initiale', 'page': 2, 'category': 'UncategorizedText', 'source': '[GU] Calcul des besoins V12.pdf'}, page_content='19/08/2011'),
 Document(metadata={'header': 'Version initiale', 'page': 2, 'category': 'UncategorizedText', 'source': '[GU] Calcul des besoins V12.pdf'}, page_content='27/09/2018'),
 Document(metadata={'header': 'Version initiale', 'page': 2, 'category': 'UncategorizedText', 'source': '[GU] Calcul des besoins V12.pdf'}, page_content='2.1, 2.4, 2.5, 3.3'

In [19]:
len(final_docs)

166

In [31]:
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [32]:
from dotenv import load_dotenv

In [22]:
# Embedding wrapper for LangChain
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3"
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 23553.69it/s]


In [23]:
dimension = 1024

In [24]:
persist_directory="embeddings_pdf_sagex3_using_recursivetextesplitter_metadata"

In [27]:
chroma_vdb=Chroma.from_documents(documents=final_docs,
                      embedding=embedding_model,
                      persist_directory=persist_directory,
                      )

In [28]:
chroma_vdb.persist()

C:\Users\tchouarr\AppData\Local\Temp\ipykernel_19944\3808622788.py:1: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  chroma_vdb.persist()


### appel de la base

In [29]:
vdb=Chroma(persist_directory=persist_directory,
        embedding_function=embedding_model)

C:\Users\tchouarr\AppData\Local\Temp\ipykernel_19944\35172397.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vdb=Chroma(persist_directory=persist_directory,


In [37]:
retriever=vdb.as_retriever(search_kwargs={"k": 30})

In [38]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [39]:
prompt = ChatPromptTemplate.from_template("""
Tu es un assistant spécialisé dans les documents administratifs français.

Réponds uniquement à partir du contexte fourni.

Contexte:
{context}

Question:
{question}
Si possible de donné le numero de la page,
Si l'information n'est pas présente dans le contexte,
réponds:
"Information non trouvée dans les documents."
""")

In [40]:
chain = prompt | llm | StrOutputParser()

In [41]:
question = "comment calculer le besoin d'un site ?"

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Le calcul du besoin d'un site peut être effectué en prenant en compte les données de plusieurs sites (ceux appartenant au groupe de site). Pour ce faire, il est possible de déclarer le groupe de site dans les données de tous les sites appartenant au groupe. Les données de consommations, du disponible, des encours fournisseurs, des encours services sont cumulées selon tous les sites du groupe.

Le calcul du besoin net est effectué comme suit :

* Besoin brut = Consommation de la période explorée / nb jours de la période explorée
* Le système prend en compte les paramètres du site enregistrés dans les valeurs paramètres du groupe XAP, tels que le nombre de jours de consommation pour le calcul du besoin net pour les articles ayant un type de délai = 1 ou 2.

Il n'y a pas de numéro de page spécifique pour cette information, car le contexte fourni ne comporte pas de pagination.


In [42]:

question = "comment calculer le besoin ?"

docs = retriever.invoke(question)

context = "\n\n".join(
    [doc.page_content for doc in docs]
)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Le calcul du besoin net est effectué comme suit :

→ Besoin brut = Consommation de la période explorée / nb jours de la période explorée

Il n'y a pas de numéro de page spécifique fourni dans le contexte, mais cette information se trouve dans la section "5. Calcul du besoin net".

Il est également important de noter que le calcul du besoin net peut varier en fonction du type de délai et des paramètres définis pour chaque article, tels que :

* XNBJRCBNS1 – Nb jr conso sans délai (CBN) pour les articles sans délai
* XNBJRCBNS2 – Nb jr conso délai 1 (CBN) pour les articles ayant un type de délai = 1 ou D1
* XNBJRCBNS3 – Nb jr conso délai 2 (CBN) pour les articles ayant un type de délai = 2 ou D2

Ces paramètres permettent de définir le nombre de jours par défaut pour le calcul du besoin net pour chaque type d'article.


In [44]:
def format_docs(docs):

    formatted = []

    for doc in docs:

        page = doc.metadata.get("page", "Unknown")
        header = doc.metadata.get("header", "Sans titre")
        source = doc.metadata.get("source", "Unknown")

        chunk = f"""
Source: {source}
Page: {page}
Section: {header}

Contenu:
{doc.page_content}
"""

        formatted.append(chunk)

    return "\n\n".join(formatted)

In [ ]:
context = format_docs(retrieved_docs)

In [45]:
question = "comment calculer le besoin ?"

docs = retriever.invoke(question)

context = format_docs(docs)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Le calcul du besoin est effectué de la manière suivante :

1. Besoin brut = Consommation de la période explorée / nb jours de la période explorée (Page 14)
2. Besoin net = Besoin brut - disponible théorique (Page 15)

Il est également possible de calculer le besoin en fonction du type de délai de l'article :
- Pour les articles sans délai (Type de délai dans la fiche article = blanc), le système prendra les paramètres définis par défaut (Page 9)
- Pour les articles avec un type de délai = 1 ou D1, le système prendra les paramètres définis par XNBJRCBNS2 (Page 5)
- Pour les articles avec un type de délai = 2 ou D2, le système prendra les paramètres définis par XNBJRCBNS3 (Page 5)

Le calcul du disponible théorique est également nécessaire pour calculer le besoin net (Page 14).

Il est important de noter que les paramètres utilisés pour le calcul des besoins peuvent varier en fonction des besoins du gestionnaire et des paramètres définis dans le système.
